# Project 2 Week 6

## Man Wayne Hui

Choose any three of the “wide” datasets identified in the Week 5 Discussion items. (You may
use your own dataset; please don’t use my Sample Post dataset, since that was used in your
Project 1 assignment!) For each of the three chosen datasets:
* Create a .CSV file (or optionally, a MySQL database!) that includes all of the information
included in the dataset. You’re encouraged to use a “wide” structure similar to how the
information appears in the discussion item, so that you can practice tidying and
transformations as described below.
* Read the information from your .CSV file into Python, and use pandas as needed to tidy
and transform your data. [Most of your grade will be based on this step!]
* Perform the analysis requested in the discussion item.
* Your code should be in an Jupyter Notebook, posted to your GitHub repository, and
should include narrative descriptions of your data cleanup work, analysis, and
conclusions.

Please include the URL for the Jupyter Notebook in your homework submission, for each of the
three chosen datasets. You may create a single Jupyter Notebook, or three separate
notebooks.

In [139]:
import pandas as pd

df = pd.read_csv("Project2.csv")

df

,Location,Sales Channel,Electronics,Apparel,Home & Kitchen,Sporting Goods
0,Downtown,In-Store,320,580,210,180
1,Downtown,Online Pickup,850,620,340,290
2,Midtown,In-Store,280,490,190,150
3,Midtown,Online Pickup,710,530,280,240
4,Suburb,In-Store,190,340,260,210
5,Suburb,Online Pickup,420,380,310,260


### Loading Dataset
Import the pandas libaray then use .read_csv() function on "Project2.csv" to convert it into a useable dataframe.

In [140]:
tidy_df = df.melt(
    id_vars=['Location', 'Sales Channel'],
    value_vars=['Electronics', 'Apparel', 'Home & Kitchen', 'Sporting Goods'],
    var_name='Product Category',
    value_name='Units Sold'
)

tidy_df


,Location,Sales Channel,Product Category,Units Sold
0,Downtown,In-Store,Electronics,320
1,Downtown,Online Pickup,Electronics,850
2,Midtown,In-Store,Electronics,280
3,Midtown,Online Pickup,Electronics,710
4,Suburb,In-Store,Electronics,190
5,Suburb,Online Pickup,Electronics,420
6,Downtown,In-Store,Apparel,580
7,Downtown,Online Pickup,Apparel,620
8,Midtown,In-Store,Apparel,490
9,Midtown,Online Pickup,Apparel,530


### Tidy Data
The original dataframe is a wide format which makes it hard to properly manage the data. In order to make it easier to manipulate, I use the .melt() function to transform it into a long format. The id_vars corrlates the entire dataframe base on Locations and Sales Channel making it easier to manipulate the data into what I want to see for the later charts.

In [141]:
## Total Units Sold by Location and Sales Channel
channel_location_sales = tidy_df.groupby(['Location', 'Sales Channel'])['Units Sold'].sum().reset_index()

display(channel_location_sales)


## Percentage Difference (Online vs In-Store) by Location
channel_diff_df = channel_location_sales.pivot(
    index='Location',
    columns='Sales Channel',
    values='Units Sold'
).rename_axis(columns=None).reset_index()

# Calculate total units sold per location
total_units = channel_diff_df['In-Store'] + channel_diff_df['Online Pickup']


# Calculate the percentage of total units sold for each sales channel by location
channel_diff_df['% In Store'] = (channel_diff_df['In-Store'] / total_units * 100).round(2)
channel_diff_df['% Online Pickup'] = (channel_diff_df['Online Pickup'] / total_units * 100).round(2)

channel_diff_df


,Location,Sales Channel,Units Sold
0,Downtown,In-Store,1290
1,Downtown,Online Pickup,2100
2,Midtown,In-Store,1110
3,Midtown,Online Pickup,1760
4,Suburb,In-Store,1000
5,Suburb,Online Pickup,1370


,Location,In-Store,Online Pickup,% In Store,% Online Pickup
0,Downtown,1290,2100,38.05,61.95
1,Midtown,1110,1760,38.68,61.32
2,Suburb,1000,1370,42.19,57.81


### Groupby
With the tidy dataset we can use groupby to sum up the values of the units sold. This will remove the product category and combine all the sales together making it easier to look at our chart to understand and determind which stores more online or in store.

### Analysis #1 (Online Vs In-Store)
The data was pivoted to compare in-store purchases against online pickup orders. Across all three locations, online pickup accounted for majority of the sales volume leading with Downtown of 61.95% then Midtown of 61.31% and lastly Suburb at 57.81% of our online sales compared to in-store stales. The trend suggests that customers located farther from the city may be more inclined to browse and purchacse in store rather than on online pickups.

In [142]:
category_location_pivot = tidy_df.pivot_table(
    values='Units Sold',
    index='Location',
    columns='Product Category',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
).rename_axis(columns=None).reset_index()

category_location_pivot


,Location,Apparel,Electronics,Home & Kitchen,Sporting Goods,Total
0,Downtown,1200,1170,550,470,3390
1,Midtown,1020,990,470,390,2870
2,Suburb,720,610,570,470,2370
3,Total,2940,2770,1590,1330,8630


### Category && Totals
Using the `pivot_table()` function on the tidy dataset to reshaped the total units sold across store locations and product categories. The `margins=True` parameter automatically calculates row and column subtotals, providing immediate insight into overall store performance and category volume.`.reset_index()` fixes the index row and presents a clean table.

### Analysis #2 (category)
Downtown made the highest overall sale volume compared to other locations with 3,390 units sold, followed by Midtown with 2,870 units sold and Suburb with 2,370 unts sold. Across all stores, apparel and electronics  drove the vast majority of sales, contributing nearly two-thirds of the 8,630 total units sold. While urban locations were heavily dominated by tech and apparel, the Suburb store showed more balanced demand with Home & Kitchen nearly matching its electronics volume.   


In [144]:
channel_category_pivot = tidy_df.pivot_table(
    values='Units Sold',
    index='Product Category',
    columns='Sales Channel',
    aggfunc='sum'
).rename_axis(columns=None).reset_index()

# Calculate the difference between Online Pickup and In-Store sales for each product category
channel_category_pivot['Difference (Online - In-Store)'] = (
    channel_category_pivot['Online Pickup'] - channel_category_pivot['In-Store']
)

channel_category_pivot

,Product Category,In-Store,Online Pickup,Difference (Online - In-Store)
0,Apparel,1410,1530,120
1,Electronics,790,1980,1190
2,Home & Kitchen,660,930,270
3,Sporting Goods,540,790,250


# Online Vs In-store based on Categories
Using `pivot_table()`, the tidy data is aggregated to summarize total `Units Sold` across each `Sales Channel` and `Product Category`. Setting `aggfunc='sum'` adds up the units sold for each channel-category pair. Finally, clearing `columns.name` removes the stacked column header label to keep the output display clean.

### Analysis 3 (Category Vs Online/In-Store)
Online Pickup generated higher sales volume across every product category compared to In-Store shopping, with the largest gap occurring in Electronics of 1,980 online vs. 790 in-store. In contrast, Apparel exhibited the most balanced demand across channels, recording 1,530 online orders and 1,410 in-person purchases. This pattern shows that customers strongly favor online orders for tech goods, where as clothing continues to drive steady balance.